In [1]:
%pwd

'/kaggle/working'

In [2]:
!git clone https://github.com/PaddlePaddle/PaddleOCR.git

Cloning into 'PaddleOCR'...
remote: Enumerating objects: 333268, done.
remote: Counting objects: 100% (105/105), done.
remote: Compressing objects: 100% (32/32), done.
remote: Total 333268 (delta 88), reused 73 (delta 73), pack-reused 333163 (from 2)
Receiving objects: 100% (333268/333268), 1.77 GiB | 33.71 MiB/s, done.
Resolving deltas: 100% (264056/264056), done.


In [3]:
%%writefile /kaggle/working/PaddleOCR/tools/infer/predict_system.py
# Copyright (c) 2020 PaddlePaddle Authors. All Rights Reserved.
#
# Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
#     http://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.
import os
import sys

__dir__ = os.path.dirname(os.path.abspath(__file__))
sys.path.append(__dir__)
sys.path.insert(0, os.path.abspath(os.path.join(__dir__, "../..")))

os.environ["FLAGS_allocator_strategy"] = "auto_growth"

import cv2
import copy
import numpy as np
import json
import time
import logging
import tools.infer.utility as utility
import tools.infer.predict_rec as predict_rec
import tools.infer.predict_det as predict_det
import tools.infer.predict_cls as predict_cls
from ppocr.utils.utility import get_image_file_list, check_and_read
from ppocr.utils.logging import get_logger
from tools.infer.utility import (
    get_rotate_crop_image,
    get_minarea_rect_crop,
)

logger = get_logger()


class TextSystem(object):
    def __init__(self, args):
        if not args.show_log:
            logger.setLevel(logging.INFO)

        self.text_detector = predict_det.TextDetector(args)
        self.text_recognizer = predict_rec.TextRecognizer(args)
        self.use_angle_cls = args.use_angle_cls
        self.drop_score = args.drop_score
        if self.use_angle_cls:
            self.text_classifier = predict_cls.TextClassifier(args)
        self.args = args
        
        # Initialize timing statistics
        self.det_time = 0
        self.cls_time = 0
        self.rec_time = 0

    def __call__(self, img, cls=True):
        if img is None:
            logger.debug("no valid image provided")
            return None, None

        ori_im = img.copy()
        
        # Time detection process
        det_start = time.time()
        dt_boxes, _ = self.text_detector(img)
        det_elapsed = time.time() - det_start
        self.det_time += det_elapsed

        if dt_boxes is None:
            return None, None

        img_crop_list = []
        dt_boxes = sorted_boxes(dt_boxes)

        for bno in range(len(dt_boxes)):
            tmp_box = copy.deepcopy(dt_boxes[bno])
            if self.args.det_box_type == "quad":
                img_crop = get_rotate_crop_image(ori_im, tmp_box)
            else:
                img_crop = get_minarea_rect_crop(ori_im, tmp_box)
            img_crop_list.append(img_crop)
        
        # Time classification process (if enabled)
        if self.use_angle_cls and cls:
            cls_start = time.time()
            img_crop_list, _, _ = self.text_classifier(img_crop_list)
            cls_elapsed = time.time() - cls_start
            self.cls_time += cls_elapsed

        # Time recognition process
        rec_start = time.time()
        rec_res, _ = self.text_recognizer(img_crop_list)
        rec_elapsed = time.time() - rec_start
        self.rec_time += rec_elapsed
        
        # Filter by score
        filter_boxes, filter_rec_res = [], []
        for box, rec_result in zip(dt_boxes, rec_res):
            text, score = rec_result[0], rec_result[1]
            if score >= self.drop_score:
                filter_boxes.append(box)
                filter_rec_res.append(rec_result)
                
        return filter_boxes, filter_rec_res
    
    def get_timing_stats(self):
        """Return timing statistics"""
        return {
            'detection_time': self.det_time,
            'classification_time': self.cls_time,
            'recognition_time': self.rec_time,
            'total_time': self.det_time + self.cls_time + self.rec_time
        }


def sorted_boxes(dt_boxes):
    """Sort text boxes in order from top to bottom, left to right"""
    num_boxes = dt_boxes.shape[0]
    sorted_boxes = sorted(dt_boxes, key=lambda x: (x[0][1], x[0][0]))
    _boxes = list(sorted_boxes)

    for i in range(num_boxes - 1):
        for j in range(i, -1, -1):
            if abs(_boxes[j + 1][0][1] - _boxes[j][0][1]) < 10 and (
                _boxes[j + 1][0][0] < _boxes[j][0][0]
            ):
                tmp = _boxes[j]
                _boxes[j] = _boxes[j + 1]
                _boxes[j + 1] = tmp
            else:
                break
    return _boxes


def process_image(text_sys, image_path):
    """Process a single image and return OCR results as JSON"""
    img, flag_gif, flag_pdf = check_and_read(image_path)
    if not flag_gif and not flag_pdf:
        img = cv2.imread(image_path)
    
    if img is None:
        logger.error(f"Failed to load image: {image_path}")
        return {"root": []}
    
    dt_boxes, rec_res = text_sys(img)
    
    if dt_boxes is None or rec_res is None:
        return {"root": []}
    
    # Build result list
    results = []
    for i in range(len(dt_boxes)):
        results.append({
            "transcription": rec_res[i][0],
            "points": np.array(dt_boxes[i]).astype(np.int32).tolist(),
        })
    
    return {"root": results}


def main(args):
    """Main function that returns JSON directly"""
    text_sys = TextSystem(args)
    
    # Warm up if needed
    if args.warmup:
        img = np.random.uniform(0, 255, [640, 640, 3]).astype(np.uint8)
        for i in range(2):  # Reduced from 10 to 2
            _ = text_sys(img)
        # Reset timing after warmup
        text_sys.det_time = 0
        text_sys.cls_time = 0
        text_sys.rec_time = 0
    
    # Process the image
    total_start = time.time()
    result = process_image(text_sys, args.image_dir)
    total_elapsed = time.time() - total_start
    
    # Get timing statistics
    timing_stats = text_sys.get_timing_stats()
    
    # Print timing information
    print("\n" + "="*50)
    print("TIMING STATISTICS")
    print("="*50)
    print(f"Detection time:       {timing_stats['detection_time']:.4f} seconds")
    if text_sys.use_angle_cls:
        print(f"Classification time:  {timing_stats['classification_time']:.4f} seconds")
    print(f"Recognition time:     {timing_stats['recognition_time']:.4f} seconds")
    print(f"Total inference time: {timing_stats['total_time']:.4f} seconds")
    print(f"Overall time:         {total_elapsed:.4f} seconds")
    print("="*50 + "\n")
    
    # Save to JSON file
    output_path = "ocr_results.json"
    with open(output_path, 'w', encoding='utf-8') as f:
        json.dump(result, f, ensure_ascii=False, indent=2)
    
    logger.info(f"OCR completed: {len(result['root'])} text regions found")
    logger.info(f"Results saved to {output_path}")
    
    return result


if __name__ == "__main__":
    args = utility.parse_args()
    result = main(args)

Overwriting /kaggle/working/PaddleOCR/tools/infer/predict_system.py


In [4]:
!python -m pip install paddlepaddle-gpu==3.2.2 -i https://www.paddlepaddle.org.cn/packages/stable/cu129/
!pip install -q paddleocr --user
!pip install -q imutils --user
!pip install -q pyclipper lmdb rapidfuzz paddleocr opencv-python --user

Looking in indexes: https://www.paddlepaddle.org.cn/packages/stable/cu129/
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 GB 928.6 kB/s eta 0:00:000:0100:03
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 65.5/65.5 kB 3.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 89.6/89.6 MB 6.9 MB/s eta 0:00:00:00:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.5/3.5 MB 15.9 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.8/10.8 MB 16.0 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 781.4/781.4 MB 2.1 MB/s eta 0:00:0000:0100:02
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 580.7/580.7 MB 1.3 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 200.9/200.9 MB 6.1 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.3/68.3 MB 11.2 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 331.8/331.8 MB 4.3 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━

In [5]:
!pwd
import os
import cv2
import json

import numpy as np
import matplotlib.pyplot as plt

from tqdm import tqdm
from PIL import Image
from imutils import perspective

# from paddleocr import PaddleOCR

/kaggle/working


In [6]:
import os
import paddle

!pip show paddleocr
print("GPU available:", paddle.device.is_compiled_with_cuda())

/usr/local/lib/python3.12/dist-packages/paddle/utils/cpp_extension/extension_utils.py:718: UserWarning: No ccache found. Please be aware that recompiling all source files may be required. You can download and install ccache from: https://github.com/ccache/ccache/blob/master/doc/INSTALL.md
  warnings.warn(warning_message)


Name: paddleocr
Version: 3.4.1
Summary: Awesome multilingual OCR and document parsing toolkits based on PaddlePaddle
Home-page: https://github.com/PaddlePaddle/PaddleOCR
Author: 
Author-email: PaddlePaddle <paddleocr@baidu.com>
License: Apache License 2.0
Location: /root/.local/lib/python3.12/site-packages
Requires: paddlex, PyYAML, requests, typing-extensions
Required-by: 
GPU available: True


In [7]:
paddle.utils.run_check()

Running verify PaddlePaddle program ... 


/usr/local/lib/python3.12/dist-packages/paddle/pir/math_op_patch.py:219: UserWarning: Value do not have 'place' interface for pir graph mode, try not to use it. None will be returned.
  warnings.warn(


PaddlePaddle works well on 1 GPU.


I0416 03:57:46.738610    55 pir_interpreter.cc:1524] New Executor is Running ...
W0416 03:57:46.740382    55 gpu_resources.cc:114] Please NOTE: device: 0, GPU Compute Capability: 7.5, Driver API Version: 13.0, Runtime API Version: 12.9
I0416 03:57:46.742990    55 pir_interpreter.cc:1547] pir interpreter is running by multi-thread mode ...
/usr/local/lib/python3.12/dist-packages/paddle/utils/cpp_extension/extension_utils.py:718: UserWarning: No ccache found. Please be aware that recompiling all source files may be required. You can download and install ccache from: https://github.com/ccache/ccache/blob/master/doc/INSTALL.md
  warnings.warn(warning_message)
/usr/local/lib/python3.12/dist-packages/paddle/utils/cpp_extension/extension_utils.py:718: UserWarning: No ccache found. Please be aware that recompiling all source files may be required. You can download and install ccache from: https://github.com/ccache/ccache/blob/master/doc/INSTALL.md
  warnings.warn(warning_message)
=============

PaddlePaddle works well on 2 GPUs.
PaddlePaddle is installed successfully! Let's start deep learning with PaddlePaddle now.


detection_model='/kaggle/working/models/detection'
recognition_model='/kaggle/working/models/recognition'

In [9]:
!python -m pip install -U scikit-image albumentations transformers accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 91.9 MB/s eta 0:00:00:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 383.7/383.7 kB 26.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 642.6/642.6 kB 34.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 393.1/393.1 MB 4.6 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 89.3 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.7/23.7 MB 65.7 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 897.7/897.7 kB 41.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 706.8/706.8 MB 2.6 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 200.2/200.2 MB 9.1 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 48.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 8.4 MB/s eta 0:00:00:00:01:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 158.2/

In [ ]:
!python PaddleOCR/tools/infer/predict_system.py \
    --det_algorithm="DB" \
    --det_model_dir="inference/det" \
    --rec_model_dir="inference/rec" \
    --image_dir="public/11.jpeg" \
    --use_gpu=False \
    --enable_mkldnn=False
    # --use_angle_cls=True

In [ ]:
!python PaddleOCR/tools/infer/predict_det.py \
    --det_algorithm="DB"\
    --det_model_dir="/kaggle/input/models/det" \
    --image_dir="/kaggle/input/testdata/test.jpg" \
    --use_gpu=False\
    --enable_mkldnn=False\
    --use_angle_cls=True

In [ ]:
def center_img(result_test):
    centers_dict = {}
    for item in result_test:
        points = item['points']
        text = item['transcription']
        x_center = (points[0][0] + points[2][0]) / 2
        y_center = (points[0][1] + points[2][1]) / 2

        # Thêm vào dictionary với key là points và value là [x_center, y_center, text]
        centers_dict[tuple(map(tuple, points))] = [x_center, y_center, text]

    return centers_dict


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.cluster import DBSCAN
from matplotlib.patches import Polygon

def expand_bboxes(bb_centers, eps):
    # Chuyển đổi dictionary bb_centers thành array để sử dụng cho DBSCAN
    bb = np.array(list(bb_centers.keys()))       # Tất cả bounding boxes
    centers = np.array([value[:2] for value in bb_centers.values()])  # Tọa độ trung tâm
    texts = [value[2] for value in bb_centers.values()]  # Văn bản

    # Áp dụng DBSCAN để phân nhóm các bounding boxes dựa trên tọa độ trung tâm
    db = DBSCAN(eps=eps, min_samples=1).fit(centers)
    labels = db.labels_

    # Tạo dictionary để lưu các nhóm bounding boxes và văn bản tương ứng
    clusters = {}
    for i, label in enumerate(labels):
        if label not in clusters:
            clusters[label] = {'bboxes': [], 'texts': []}
        clusters[label]['bboxes'].append(bb[i])
        clusters[label]['texts'].append(texts[i])

    # Kết hợp bounding boxes và text cho mỗi nhóm
    combined_results = []
    for label in clusters:
        bboxes = clusters[label]['bboxes']
        texts = clusters[label]['texts']

        # Kết hợp văn bản và bounding boxes
        text_str = ' '.join(texts)  # Nối tất cả văn bản lại với nhau
        combined_results.append([text_str, bboxes])

    return combined_results

In [ ]:
import json
def run(img_path, img_name):
    run_predict(img_path)
    with open('ocr_results.json', 'r') as f:
        result_test = json.load(f)
    output_list = []
    if result_test == []: 
        with open(f"{img_name}.json", 'w', encoding='utf-8') as json_file:
             json.dump(output_list, json_file, ensure_ascii=False, indent=4)
    else:           
        label_dict = expand_bboxes(center_img(result_test), eps=100)
        for item in label_dict:
            output_list.append(item[0])
        with open(f"{img_name}.json", 'w', encoding='utf-8') as json_file:
            json.dump(output_list, json_file, ensure_ascii=False, indent=4)

In [ ]:
import os
import re
from tqdm.notebook import tqdm  # Use this in Jupyter environments



def natural_sort_key(s):
    return [int(c) if c.isdigit() else c.lower() for c in re.split(r'(\d+)', s)]

input_path = "/kaggle/input/testdata"
output_path = "/keyframe_ocr"

if not os.path.exists(output_path):
    os.makedirs(output_path)

for root, dirs, files in os.walk(input_path):
    dirs.sort(key=natural_sort_key)
    for vxx_dir in tqdm(dirs, desc="Processing directories", leave=False):
        #if vxx_dir != "V001": break
        vxx_path = os.path.join(root, vxx_dir)
        vxx_output_path = os.path.join(output_path, os.path.relpath(root, input_path), vxx_dir)

        if not os.path.exists(vxx_output_path):
            os.makedirs(vxx_output_path)

        files = sorted(os.listdir(vxx_path))
        for file in tqdm(files, desc=f"Processing files in {vxx_dir}", leave=False):
            #if file != "000000.jpg": continue
            if file.lower().endswith('.jpg'):
                file_path = os.path.join(vxx_path, file)
                output_file = os.path.join(vxx_output_path, file[:-4])
                run(file_path, output_file)

print("Processing complete.")


In [ ]:
from PIL import Image, ImageDraw, ImageFont
import numpy as np

def draw_boxes(image_path, ocr_results, output_path, flag):
    with Image.open(image_path) as img:
        draw = ImageDraw.Draw(img)
        try:
            font = ImageFont.truetype("/content/1942.ttf", 40)
        except IOError:
            font = ImageFont.load_default()

        if flag == 1:
            for item in ocr_results:
                flat_bbox = [coord for point in item['points'] for coord in point]
                draw.polygon(flat_bbox, outline=(255, 0, 0), width=2)
        else:
            for item in ocr_results:
                all_points = np.vstack(item[1])
                text = item[0]
                print(text)

                # Calculate the bounding box
                x_min, y_min = np.min(all_points, axis=0)
                x_max, y_max = np.max(all_points, axis=0)

                expanded_bbox = np.array([[x_min, y_min], [x_max, y_min], [x_max, y_max], [x_min, y_max]])

                # Flatten the bounding box for drawing
                flat_bbox = expanded_bbox.flatten().tolist()
                draw.polygon(flat_bbox, outline=(255, 0, 0), width=2)

                text_position = (x_min, max(0, y_min - 20))
                draw.text(text_position, text, font=font, fill=(0, 255, 0))  # Green text

        img.save(output_path)

        img.show()  # This will open the image with the drawn text and boxes

In [ ]:
def run_predict(image_dir):
    #os.chdir('/kaggle/working/ppocr-vietnamese')
    command = f"""
    python PaddleOCR/tools/infer/predict_system.py \\
        --image_dir="{image_dir}" \\
        --det_model_dir="inference/det" \\
        --det_algorithm="DB" \\
        --rec_image_shape="3, 48, 320" \\
        --rec_model_dir="inference/rec" \\
        --rec_char_dict_path="vn_dictionary.txt" \\
        --vis_font_path="vietnam-light.ttf" \\
        --drop_score 0.6 \
        --use_gpu USE_GPU
    """
    os.system(command)

In [ ]:
!rm -r inference_results

In [ ]:
# First, uncomment and use the draw_boxes function
from PIL import Image, ImageDraw, ImageFont
import numpy as np

def draw_boxes(image_path, ocr_results, output_path, flag):
    with Image.open(image_path) as img:
        draw = ImageDraw.Draw(img)
        try:
            font = ImageFont.truetype("vietnam-light.ttf", 40)  # Use available font
        except IOError:
            font = ImageFont.load_default()

        if flag == 1:
            for item in ocr_results:
                flat_bbox = [coord for point in item['points'] for coord in point]
                draw.polygon(flat_bbox, outline=(255, 0, 0), width=2)
        else:
            for item in ocr_results:
                all_points = np.vstack(item[1])
                text = item[0]
                print(text)

                # Calculate the bounding box
                x_min, y_min = np.min(all_points, axis=0)
                x_max, y_max = np.max(all_points, axis=0)

                expanded_bbox = np.array([[x_min, y_min], [x_max, y_min], [x_max, y_max], [x_min, y_max]])

                # Flatten the bounding box for drawing
                flat_bbox = expanded_bbox.flatten().tolist()
                draw.polygon(flat_bbox, outline=(255, 0, 0), width=2)

                text_position = (x_min, max(0, y_min - 20))
                draw.text(text_position, text, font=font, fill=(0, 255, 0))

        img.save(output_path)
        img.show()

In [ ]:
!rm ocr_result.json

In [ ]:
import json
import subprocess
import sys
import re

def run_ocr(image_path):
    """Run OCR and return results as JSON with labels and bounding boxes."""
    
    command = [
        sys.executable,  # Use current Python interpreter
        "/kaggle/working/PaddleOCR/tools/infer/predict_system.py",
        "--det_algorithm=DB",
        "--det_model_dir=/kaggle/input/models/det",
        "--rec_model_dir=/kaggle/input/recognize/rec",
        f"--image_dir={image_path}",
        "--use_gpu=True",
        "--rec_char_dict_path=/kaggle/input/utils-model/vn_dictionary.txt" 
        # "--enable_mkldnn=False",
        # "--show_log=False",  # Disable verbose logging for speed
        # "--warmup=False",     # Skip warmup for faster execution
        # "--rec_batch_num=32",
        # "--cpu_threads=16",
        
    ]
    
    # Run with subprocess for better control and error handling
    try:
        result = subprocess.run(
            command,
            capture_output=True,
            text=True,
            # timeout=30  # 30 second timeout
        )
        
        if result.returncode != 0:
            print(f"OCR Error: {result.stderr}")
            return [], {}
        
        # Extract timing information from stdout
        timing_info = extract_timing_info(result.stdout)
            
    except subprocess.TimeoutExpired:
        print("OCR process timeout after 30 seconds")
        return [], {}
    except Exception as e:
        print(f"OCR failed: {e}")
        return [], {}
    
    # Load the JSON results
    try:
        with open('ocr_results.json', 'r', encoding='utf-8') as f:
            data = json.load(f)
    except FileNotFoundError:
        print("OCR results file not found")
        return [], {}
    except json.JSONDecodeError:
        print("Invalid JSON in OCR results")
        return [], {}
    
    # Transform the data to add label and bounding_box fields
    results = []
    for item in data.get('root', []):
        points = item['points']
        results.append({
            'transcription': item['transcription'],
            'label': item['transcription'],
            'points': points,
            'bounding_box': {
                'top_left': points[0],
                'top_right': points[1],
                'bottom_right': points[2],
                'bottom_left': points[3]
            }
        })
    
    return results, timing_info


def extract_timing_info(stdout):
    """Extract timing information from stdout."""
    timing_info = {}
    
    # Patterns to match timing lines
    patterns = {
        'detection': r'Detection time:\s+([\d.]+) seconds',
        'classification': r'Classification time:\s+([\d.]+) seconds',
        'recognition': r'Recognition time:\s+([\d.]+) seconds',
        'total_inference': r'Total inference time:\s+([\d.]+) seconds',
        'overall': r'Overall time:\s+([\d.]+) seconds'
    }
    
    for key, pattern in patterns.items():
        match = re.search(pattern, stdout)
        if match:
            timing_info[key] = float(match.group(1))
    
    return timing_info


def print_timing_info(timing_info):
    """Print timing information in a formatted way."""
    if not timing_info:
        return
    
    print("\n" + "="*50)
    print("TIMING STATISTICS")
    print("="*50)
    
    if 'detection' in timing_info:
        print(f"Detection time:       {timing_info['detection']:.4f} seconds")
    
    if 'classification' in timing_info:
        print(f"Classification time:  {timing_info['classification']:.4f} seconds")
    
    if 'recognition' in timing_info:
        print(f"Recognition time:     {timing_info['recognition']:.4f} seconds")
    
    if 'total_inference' in timing_info:
        print(f"Total inference time: {timing_info['total_inference']:.4f} seconds")
    
    if 'overall' in timing_info:
        print(f"Overall time:         {timing_info['overall']:.4f} seconds")
    
    print("="*50 + "\n")


# Usage:
if __name__ == "__main__":
    ocr_results, timing_info = run_ocr("/kaggle/input/testdata/test.jpg")
    
    # Print timing information
    print_timing_info(timing_info)
    
    if ocr_results:
        print(f"✓ Found {len(ocr_results)} text regions\n")
        
        # Print results
        for i, item in enumerate(ocr_results, 1):
            print(f"{i}. Label: {item['label']}")
            print(f"   Bounding Box: {item['bounding_box']}")
            print("---")
    else:
        print("No text detected or OCR failed")

In [ ]:
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

In [2]:
import json
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from typing import List, Dict, Optional
import time

# ==================== 1. INITIALIZE LOCAL LLM ====================
# We do this globally so the model doesn't reload on every function call.

MODEL_ID = "Qwen/Qwen2.5-3B-Instruct"

print(f"Loading {MODEL_ID} into Kaggle GPU...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16, # Uses float16 to save memory and speed up inference
    device_map="auto"          # Automatically offloads to the Kaggle GPU
)
print("✓ Local model loaded successfully!")

# ==================== 2. PROCESSING FUNCTION ====================

def ocr_to_structured_json(ocr_results: List[Dict]) -> Optional[Dict]:
    """
    Convert OCR results to structured medicine invoice JSON using a local LLM.
    """
    if not ocr_results:
        print("Error: No OCR results to process")
        return None
    
    # Prepare the OCR text
    ocr_text = "\n".join([f"- {item['label']}" for item in ocr_results])
    
    # Create the prompt for the LLM
    prompt = f"""You are an expert at extracting structured data from Vietnamese medicine invoices/prescriptions.
Given the following OCR text from a medicine invoice, extract and return a JSON object.
Grammar & Spelling Correction: During extraction, identify and correct any Vietnamese spelling errors or grammatical inconsistencies caused by OCR inaccuracies (e.g., wrong tone marks, missing characters, or incorrect medical terms). Ensure the extracted values are professionally formatted and linguistically correct in a medical context.

Extract and return a JSON object with this exact structure:

{{
  "patient_name": "full name of patient or null if not found",
  "doctor_name": "name of doctor or null if not found",
  "clinic_hospital": "name of clinic or hospital or null if not found",
  "date": "date of prescription (DD/MM/YYYY) or null if not found",
  "diagnosis": "diagnosis or condition or null if not found",
  "medicines": [
    {{
      "name": "medicine name",
      "dosage": "dosage amount (e.g., 500mg, 10ml)",
      "quantity": "quantity prescribed",
      "frequency": "how often to take (e.g., 2 times/day)",
      "duration": "duration of treatment",
      "instructions": "special instructions"
    }}
  ],
  "notes": "any additional notes or warnings or null if not found"
}}

OCR Text:
{ocr_text}

IMPORTANT: 
- Return ONLY valid JSON, no explanation or markdown formatting
- Use null for fields that cannot be found in the text
- Extract ALL medicines found in the prescription
- Keep Vietnamese characters intact"""

    # Format using the model's specific chat template
    messages = [
        {"role": "system", "content": "You are a helpful data extraction assistant. You only output valid JSON."},
        {"role": "user", "content": prompt}
    ]
    
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )
    
    # Tokenize and move to GPU
    model_inputs = tokenizer([text], return_tensors="pt").to(model.device)
    
    print("Generating structured data locally...")
    try:
        # Generate the response
        generated_ids = model.generate(
            **model_inputs,
            max_new_tokens=1500,
            temperature=0.1,  # Keep temperature low for deterministic JSON output
            do_sample=True,
        )
        
        # Slice the output to only get the newly generated tokens (ignore the prompt tokens)
        generated_ids = [
            output_ids[len(input_ids):] for input_ids, output_ids in zip(model_inputs.input_ids, generated_ids)
        ]
        
        llm_response = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]
        
        # Clean up the response (your existing robust parsing logic)
        llm_response = llm_response.strip()
        
        if llm_response.startswith("```json"):
            llm_response = llm_response[7:]
        elif llm_response.startswith("```"):
            llm_response = llm_response[3:]
        
        if llm_response.endswith("```"):
            llm_response = llm_response[:-3]
            
        structured_data = json.loads(llm_response.strip())
        
        print("✓ Successfully converted OCR to structured data")
        return structured_data
        
    except json.JSONDecodeError as e:
        print(f"JSON parsing error: {e}")
        print(f"LLM Response snippet: {llm_response[:200]}...")
        return None
            
    except Exception as e:
        print(f"Unexpected error: {e}")
        return None


# ==================== USAGE ====================

if __name__ == "__main__":
    # Mocking your run_ocr output for testing purposes
    # Replace this with your actual run_ocr call: 
    # ocr_results, timing_info = run_ocr("/kaggle/input/testdata/11.jpeg") 
    
    ocr_results = [
        {"label": "Phòng khám Đa khoa Quốc tế"},
        {"label": "Bác sĩ: Nguyễn Văn A"},
        {"label": "Bệnh nhân: Trần Thị B"},
        {"label": "Chẩn đoán: Viêm họng cấp"},
        {"label": "1. Paracetamol 500mg - 10 viên - Ngày uống 2 lần"},
        {"label": "Ngày: 20/10/2023"}
    ]
    
    print("\nConverting to structured data...")
    structured_invoice = ocr_to_structured_json(ocr_results)
    
    if structured_invoice:
        print("\n" + "="*50)
        print("STRUCTURED MEDICINE INVOICE")
        print("="*50)
        print(json.dumps(structured_invoice, indent=2, ensure_ascii=False))

Loading Qwen/Qwen2.5-3B-Instruct into Kaggle GPU...


`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

✓ Local model loaded successfully!

Converting to structured data...
Generating structured data locally...
✓ Successfully converted OCR to structured data

STRUCTURED MEDICINE INVOICE
{
  "patient_name": "Trần Thị B",
  "doctor_name": "Nguyễn Văn A",
  "clinic_hospital": "Phòng khám Đa khoa Quốc tế",
  "date": "20/10/2023",
  "diagnosis": "Viêm họng cấp",
  "medicines": [
    {
      "name": "Paracetamol",
      "dosage": "500mg",
      "quantity": "10 viên",
      "frequency": "2 lần/ngày",
      "duration": null,
      "instructions": null
    }
  ],
  "notes": null
}


In [1]:
# ==============================================================================
# UPDATED BRIDGE CELL (Auto-cleanup + Robust Tunnel)
# ==============================================================================

# 1. Install Bridge Dependencies (if not already installed)
!pip install -q fastapi uvicorn python-multipart

# 2. Cleanup: Kill any processes running on port 8000 or old tunnels
!fuser -k 8000/tcp || true
!pkill cloudflared || true

import threading
import time
import subprocess
import re
from fastapi import FastAPI, File, UploadFile
from fastapi.responses import JSONResponse

app = FastAPI()

@app.post("/ocr")
async def process_ocr(file: UploadFile = File(...)):
    file_path = "/kaggle/working/temp_scan.jpg"
    with open(file_path, "wb") as buffer:
        buffer.write(await file.read())
    
    try:
        # Assuming run_ocr and ocr_to_structured_json are defined above
        ocr_results, timing = run_ocr(file_path)
        if not ocr_results:
            return JSONResponse(status_code=500, content={"error": "OCR failed"})
            
        structured_data = ocr_to_structured_json(ocr_results, OPENROUTER_API_KEY)
        return structured_data
    except Exception as e:
        return JSONResponse(status_code=500, content={"error": str(e)})

@app.get("/health")
def health():
    return {"status": "ready"}

def run_server():
    import uvicorn
    # Using 0.0.0.0 to ensure it's accessible within the local Kaggle container
    uvicorn.run(app, host="0.0.0.0", port=8000, log_level="error")

# Start API in background
threading.Thread(target=run_server, daemon=True).start()
time.sleep(3)

# 3. Start Public Tunnel
print("\n" + "="*60)
print("  ESTABLISHING PUBLIC URL... PLEASE WAIT")
print("="*60)

tunnel_proc = subprocess.Popen(
    ["cloudflared", "tunnel", "--url", "http://localhost:8000"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True
)

# Robust URL Extraction
found_url = False
try:
    for line in tunnel_proc.stdout:
        # Standard Cloudflare log format check
        if "trycloudflare.com" in line:
            # Match any https URL ending in trycloudflare.com
            match = re.search(r'https://[a-zA-Z0-9-]+\.trycloudflare\.com', line)
            if match:
                url = match.group(0)
                print(f"\n🚀 OCR GATEWAY IS LIVE!")
                print(f"👉 COPY THIS URL TO YOUR .env: {url}")
                print("\nKeep this cell running. Logs will appear below:")
                print("-" * 60)
                found_url = True
        
        # Print logs after URL is found so you can see incoming requests
        if found_url:
            if "GET /health" not in line: # Filter out health checks if annoying
                print(line.strip())
except KeyboardInterrupt:
    print("\nStopping tunnel...")
    tunnel_proc.terminate()



  ESTABLISHING PUBLIC URL... PLEASE WAIT


FileNotFoundError: [Errno 2] No such file or directory: 'cloudflared'